# 11장. LoRA와 QLoRA

10장의 미세 조정을 모델 전체가 아니라 작은 행렬 몇 개만 학습시켜 다시 한다. LoRA를 직접 구현해 PEFT 라이브러리와 맞추고, 순위와 붙일 층을 바꾸고, 4비트로 양자화한 모델 위에서 학습시키는 QLoRA까지 잰다.

**준비:** `pip install "transformers<5" "peft==0.21.2" "accelerate==1.15.0" huggingface_hub`. 첫 셀이 Qwen3-0.6B(약 1.5GB)를 `data/models/`에 받는다. NSMC의 `ratings_train.txt`, `ratings_test.txt`도 필요하다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 64분쯤 걸렸다.

## 11.1 LoRA 직접 만들기

In [ ]:
import os, re, json, math, time, sys, copy, random, platform, warnings
import numpy as np
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers, peft
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = snapshot_download("Qwen/Qwen3-0.6B", local_dir="data/models/Qwen__Qwen3-0.6B",
                         allow_patterns=["*.json", "*.txt", "*.md", "LICENSE*", "*.safetensors"])
tok = AutoTokenizer.from_pretrained(PATH)
END = tok.convert_tokens_to_ids("<|im_end|>")
load = lambda: AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float32)
print(sys.version.split()[0], torch.__version__, transformers.__version__, peft.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 10장과 같은 자료: NSMC 영화평으로 만든 긍정·부정 지시 자료(학습 400, 검증 300, 시험 500)
def read(name):
    rows = [line.split("\t") for line in open(name, encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
    return [(text.strip(), int(label)) for _, text, label in rows if 5 <= len(text.strip()) <= 80]

pick = random.Random(0)
train_all, test_all = read("ratings_train.txt"), read("ratings_test.txt")
pick.shuffle(train_all); pick.shuffle(test_all)
balance = lambda rows, n: [r for r in rows if r[1] == 1][:n // 2] + [r for r in rows if r[1] == 0][:n // 2]
TRAIN = balance(train_all[:4000], 400)
VAL = balance(train_all[4000:8000], 300)
TEST = balance(test_all, 500)
pick.shuffle(TRAIN)
WORD = {1: "긍정", 0: "부정"}
ASK = "다음 영화평이 긍정이면 '긍정', 부정이면 '부정'이라고만 답하세요.\n영화평: {}"

def prompt_ids(question):
    """질문을 9장의 채팅 템플릿(생각 모드 끔)에 넣은 토큰."""
    return tok.apply_chat_template([{"role": "user", "content": question}], tokenize=True,
                                   add_generation_prompt=True, enable_thinking=False)

def example(text, label):
    """학습 예 하나. 답('긍정'이나 '부정')과 끝 토큰에만 손실을 준다(10장의 '답만')."""
    p = prompt_ids(ASK.format(text))
    a = tok(WORD[label], add_special_tokens=False)["input_ids"] + [END]
    return p + a, [-100] * len(p) + a

print(len(TRAIN), len(VAL), len(TEST), "학습 예 하나의 토큰:", len(example(*TRAIN[0])[0]))

In [ ]:
# 10장과 같은 평가: 시험 500개 정확도, 형식, 상식 20문항, 다른 지시 다섯 개, 검증 답 손실
QA = [("대한민국의 수도는 어디인가요?", ["서울"]), ("일본의 수도는 어디인가요?", ["도쿄"]), ("물의 화학식은 무엇인가요?", ["H2O", "H₂O"]),
      ("1년은 몇 개월인가요?", ["12개월", "12 개월", "12달"]), ("세종대왕이 만든 문자는 무엇인가요?", ["한글", "훈민정음"]),
      ("태양계에서 가장 큰 행성은 무엇인가요?", ["목성"]), ("지구에서 가장 넓은 바다는 무엇인가요?", ["태평양"]), ("7 곱하기 8은 얼마인가요?", ["56"]),
      ("100에서 37을 빼면 얼마인가요?", ["63"]), ("남한에서 가장 높은 산은 무엇인가요?", ["한라산"]), ("임진왜란이 일어난 해는 언제인가요?", ["1592"]),
      ("삼각형 세 내각의 합은 몇 도인가요?", ["180"]), ("사과를 영어로 하면 무엇인가요?", ["apple", "Apple"]),
      ("서울을 가로질러 흐르는 큰 강은 무엇인가요?", ["한강"]), ("대한민국 국기의 이름은 무엇인가요?", ["태극기"]),
      ("1킬로미터는 몇 미터인가요?", ["1000", "1,000"]), ("식물 세포에서 광합성을 하는 기관은 무엇인가요?", ["엽록체"]),
      ("컴퓨터에서 1바이트는 몇 비트인가요?", ["8비트", "8 비트", "8개"]), ("소설 '어린 왕자'를 쓴 작가는 누구인가요?", ["생텍쥐페리"]),
      ("빛의 삼원색은 빨강, 초록과 무엇인가요?", ["파랑", "파란"])]
OTHER = ["다음 영화평을 영어로 번역하세요.\n영화평: 배우들의 연기가 정말 좋았다", "다음 영화평에서 배우 이름을 찾아 쓰세요.\n영화평: 송강호 연기는 최고였다",
         "다음 영화평을 한 문장으로 요약하세요.\n영화평: 처음에는 지루했지만 후반부 반전이 대단해서 끝까지 몰입했다",
         "'영화'로 시작하는 짧은 시를 한 줄 쓰세요.", "다음 문장의 띄어쓰기를 고치세요.\n문장: 영화가너무재미있었다"]

def generate(model, ids, n):
    x = torch.tensor([ids])
    with torch.inference_mode():
        out = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=n, do_sample=False, eos_token_id=[END, tok.eos_token_id],
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip()

def answer_loss(model, rows):
    total = n = 0
    model.eval()
    with torch.inference_mode():
        for text, label in rows:
            ids, labels = example(text, label)
            out = model(input_ids=torch.tensor([ids])).logits[0, :-1]
            y = torch.tensor(labels[1:])
            total += F.cross_entropy(out, y, ignore_index=-100, reduction="sum").item(); n += (y != -100).sum().item()
    return total / n

def evaluate(model, show=False):
    model.eval()
    right = exact = 0
    for text, label in TEST:
        out = generate(model, prompt_ids(ASK.format(text)), 8)
        right += out.startswith(WORD[label]); exact += out in ("긍정", "부정")
    qa = [generate(model, prompt_ids(q), 48) for q, _ in QA]
    other = [generate(model, prompt_ids(q), 48) for q in OTHER]
    r = {"acc": right / len(TEST), "exact": exact / len(TEST),
         "qa": sum(any(k in a for k in keys) for a, (_, keys) in zip(qa, QA)) / len(QA),
         "qa_pn": sum(a in ("긍정", "부정") for a in qa) / len(QA), "other_pn": sum(a in ("긍정", "부정") for a in other) / len(OTHER),
         "val": answer_loss(model, VAL)}
    if show:
        r["qa_text"], r["other_text"] = qa[:3], other
    return r

brief = lambda r: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if not k.endswith("text")}
lap = time.perf_counter()
before = evaluate(load())
print("학습 전:", brief(before), f"{time.perf_counter() - lap:.0f}초")

In [ ]:
class LoRALinear(torch.nn.Module):
    """얼린 선형층 base 옆에 작은 행렬 A(r×입력)와 B(출력×r)를 붙인다. y = base(x) + (alpha/r)·(x Aᵀ) Bᵀ"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = torch.nn.Parameter(torch.empty(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))   # B가 0이라 처음에는 원래 모델과 똑같다
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))           # nn.Linear의 기본 초기화와 같다
    def forward(self, x):
        return self.base(x) + (x @ self.A.T) @ self.B.T * self.scale

ATTN = ("q_proj", "k_proj", "v_proj", "o_proj")
MLP = ("gate_proj", "up_proj", "down_proj")
FROZEN = (torch.nn.Linear,)          # LoRA를 붙일 수 있는 층(11.4절에서 4비트 층을 더한다)

def add_lora(model, targets, r, alpha=None):
    """이름이 targets인 선형층에 LoRA를 붙이고, LoRA 행렬만 학습되게 한다. alpha를 주지 않으면 2r(배율 2)."""
    alpha = alpha or 2 * r
    for p in model.parameters():
        p.requires_grad_(False)
    for mod in list(model.modules()):
        for name, sub in list(mod.named_children()):
            if name in targets and isinstance(sub, FROZEN):
                setattr(mod, name, LoRALinear(sub, r, alpha))
    return model

def merge(model):
    """LoRA를 가중치에 합친다: W ← W + (alpha/r)·B A. 합친 뒤에는 원래 구조와 같다."""
    for mod in list(model.modules()):
        for name, sub in list(mod.named_children()):
            if isinstance(sub, LoRALinear):
                with torch.no_grad():
                    sub.base.weight += (sub.B @ sub.A) * sub.scale
                setattr(mod, name, sub.base)
    return model

count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
N_ALL = sum(p.numel() for p in load().parameters())
x = torch.tensor([prompt_ids(ASK.format(TEST[0][0]))])
with torch.inference_mode():
    ref = load()(x).logits
torch.manual_seed(0)
m = add_lora(load(), ATTN, 8)
with torch.no_grad():
    print(f"학습할 파라미터 {count(m):,}개(전체 {N_ALL:,}개의 {count(m) / N_ALL:.2%})")
    print("붙인 직후 원래 모델과의 출력 차이:", (m(x).logits - ref).abs().max().item())
    for mod in m.modules():                                      # 학습이 끝난 것처럼 B에 작은 값을 넣어 본다
        if isinstance(mod, LoRALinear):
            torch.nn.init.normal_(mod.B, std=0.02)
    y = m(x).logits
    AB = {n: (mod.A.clone(), mod.B.clone()) for n, mod in m.named_modules() if isinstance(mod, LoRALinear)}
    print("B를 바꾼 뒤 원래 모델과의 출력 차이:", round((y - ref).abs().max().item(), 4))
    print("합친 모델과의 출력 차이:", (merge(m)(x).logits - y).abs().max().item())
del m

In [ ]:
from peft import LoraConfig, get_peft_model
cfg = LoraConfig(r=8, lora_alpha=16, target_modules=list(ATTN), lora_dropout=0.0, bias="none")
pm = get_peft_model(load(), cfg)
print("PEFT 학습할 파라미터:", f"{pm.get_nb_trainable_parameters()[0]:,}개")
name = "base_model.model.model.layers.0.self_attn.q_proj"
lay = dict(pm.named_modules())[name]
print("PEFT의 층:", type(lay).__name__, "A", tuple(lay.lora_A["default"].weight.shape), "B", tuple(lay.lora_B["default"].weight.shape),
      "배율", lay.scaling["default"], "B가 모두 0:", bool((lay.lora_B["default"].weight == 0).all()))
with torch.no_grad():                                            # 직접 만든 LoRA의 A, B를 PEFT에 그대로 옮긴다
    for n, mod in pm.named_modules():
        key = n.removeprefix("base_model.model.")
        if key in AB:
            mod.lora_A["default"].weight.copy_(AB[key][0]); mod.lora_B["default"].weight.copy_(AB[key][1])
    print("PEFT와 직접 구현의 출력 차이:", (pm(x).logits - y).abs().max().item())
del pm

## 11.2 학습할 파라미터와 메모리

In [ ]:
shapes = {}                                                      # 층 이름 → (입력, 출력), 한 블록 것만
for n, mod in load().model.layers[0].named_modules():
    if isinstance(mod, torch.nn.Linear):
        shapes[n.split(".")[-1]] = (mod.in_features, mod.out_features)
print("한 블록의 선형층:", shapes, "블록 수", 28)
lora_params = lambda targets, r: 28 * sum(r * (shapes[t][0] + shapes[t][1]) for t in targets)
SETS = {"q·v": ("q_proj", "v_proj"), "어텐션 넷": ATTN, "MLP 셋": MLP, "일곱 모두": ATTN + MLP}
for k, t in SETS.items():
    print(f"{k:6}", "  ".join(f"r={r}: {lora_params(t, r):>10,}({lora_params(t, r) / N_ALL:.2%})" for r in [1, 8, 64]))
GB = 1e9
book = lambda trainable: {"가중치": 4 * N_ALL / GB, "기울기": 4 * trainable / GB, "AdamW 상태": 8 * trainable / GB}
for k, n in [("전체 미세 조정", N_ALL), ("LoRA 어텐션 r=8", lora_params(ATTN, 8))]:
    b = book(n)
    print(f"{k}: " + ", ".join(f"{a} {v:.3f}GB" for a, v in b.items()) + f", 합계 {sum(b.values()):.2f}GB")

## 11.3 순위와 붙일 층

In [ ]:
def lora_model(targets, r, seed=0, quant=False):
    torch.manual_seed(seed)                                       # A의 초기값을 정한다
    model = load()
    if quant:
        model = quantize(model)                                  # 11.4절
    return add_lora(model, targets, r)

def train(model, epochs=3, lr=1e-4, batch=8, seed=0, track=False):
    """학습할 파라미터(LoRA 행렬)만 AdamW로 학습시킨다. 반복과 배치는 10장과 같다."""
    model.train()
    data = [example(t, y) for t, y in TRAIN]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    log = []
    for ep in range(1, epochs + 1):
        order = random.Random(seed + ep).sample(range(len(data)), len(data))
        lap, seen, tot = time.perf_counter(), 0, 0.0
        model.train()
        for k in range(0, len(order), batch):
            rows = [data[i] for i in order[k:k + batch]]
            L = max(len(r[0]) for r in rows)
            ids = torch.tensor([r[0] + [tok.pad_token_id] * (L - len(r[0])) for r in rows])
            lab = torch.tensor([r[1] + [-100] * (L - len(r[1])) for r in rows])
            mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rows])
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(rows); seen += len(rows)
        entry = {"epoch": ep, "train": tot / seen, "sec": time.perf_counter() - lap}
        if track or ep == epochs:
            entry.update(evaluate(model))
        log.append(entry)
        print(brief(entry), flush=True)
    return log

runs = {}
model = lora_model(ATTN, 8)
runs["어텐션 r=8"] = train(model, track=True)
for q in [q for q, _ in QA[:3]] + OTHER:                       # 학습 뒤 모델의 답을 몇 개 본다
    print(f"  {q[:28].replace(chr(10), ' ')} → {generate(model, prompt_ids(q), 48)[:50]}")
del model

In [ ]:
# 같은 층(어텐션 넷)에 순위만 바꿔 붙인다. alpha = 2r로 두어 배율은 2로 같다.
for r in [1, 64]:
    model = lora_model(ATTN, r)
    print(f"r={r}: 학습할 파라미터 {count(model):,}개")
    runs[f"어텐션 r={r}"] = train(model)
    del model

In [ ]:
# 순위는 8로 두고 붙일 층을 바꾼다
for name, targets in [("MLP 셋", MLP), ("일곱 모두", ATTN + MLP)]:
    model = lora_model(targets, 8)
    print(f"{name} r=8: 학습할 파라미터 {count(model):,}개")
    runs[f"{name} r=8"] = train(model)
    del model

## 11.4 4비트 모델 위에서 학습하기: QLoRA

In [ ]:
class Linear4(torch.nn.Module):
    """8장의 대칭 4비트 양자화(정수 -7~7, 입력 방향 64개 묶음마다 배율)로 담은 얼린 선형층.
    정수는 int8 한 바이트에 하나씩 담는다(두 개씩 묶으면 절반). 쓸 때마다 32비트로 되돌려 곱한다."""
    def __init__(self, weight, group=64):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        g = weight.detach().reshape(-1, group)
        scale = (g.abs().amax(dim=1, keepdim=True) / 7).clamp(min=1e-12)
        self.register_buffer("q", torch.round(g / scale).clamp(-7, 7).to(torch.int8))
        self.register_buffer("s", scale)
    @property
    def weight(self):
        return (self.q.float() * self.s).reshape(self.out_features, self.in_features)
    recompute = False                                            # 11.5절: True면 되돌린 가중치를 역전파용으로 남기지 않는다
    def forward(self, x):
        if self.recompute:
            return Dequant.apply(x, self.q, self.s, (self.out_features, self.in_features))
        return F.linear(x, self.weight)

class Dequant(torch.autograd.Function):
    """4비트 정수에서 되돌린 가중치로 곱하되, 그 가중치를 남겨 두지 않고 역전파 때 정수에서 다시 되돌린다."""
    @staticmethod
    def forward(ctx, x, q, s, shape):
        ctx.save_for_backward(q, s); ctx.shape = shape
        return F.linear(x, (q.float() * s).reshape(shape))
    @staticmethod
    def backward(ctx, grad):
        q, s = ctx.saved_tensors
        return grad @ (q.float() * s).reshape(ctx.shape), None, None, None

FROZEN = (torch.nn.Linear, Linear4)

def quantize(model):
    """트랜스포머 블록의 선형층 일곱 개를 4비트로 바꾼다(토큰 표와 출력층은 32비트 그대로)."""
    for layer in model.model.layers:
        for mod in [layer.self_attn, layer.mlp]:
            for name, sub in list(mod.named_children()):
                if isinstance(sub, torch.nn.Linear):
                    setattr(mod, name, Linear4(sub.weight))
    return model

q4 = quantize(load())
n4 = sum(m.q.numel() for m in q4.modules() if isinstance(m, Linear4))
before4 = evaluate(q4)
print(f"4비트로 바꾼 가중치 {n4:,}개({n4 / N_ALL:.1%}), 학습 전 4비트 모델:", brief(before4))
del q4

model = lora_model(ATTN, 8, quant=True)
runs["QLoRA 어텐션 r=8"] = train(model)

# 학습한 어댑터를 쓰는 세 가지 방법
AB = {n: (mod.A.detach().clone(), mod.B.detach().clone()) for n, mod in model.named_modules() if isinstance(mod, LoRALinear)}
def with_adapter(m):
    """m의 같은 이름 층에 학습한 A, B를 붙인다."""
    for n, (A, B) in AB.items():
        parent, child = n.rsplit(".", 1)
        lay = LoRALinear(m.get_submodule(n), 8, 16)
        lay.A.data.copy_(A); lay.B.data.copy_(B)
        setattr(m.get_submodule(parent), child, lay)
    return m
use = {}
fp = merge(with_adapter(load()))                                  # ① 원래 32비트 모델에 붙여 합친다
use["32비트 원본에 합침"] = evaluate(fp); del fp
m4 = with_adapter(quantize(load()))                              # ② 4비트로 되돌린 가중치에 합친 뒤 다시 4비트로 담는다
for mod in list(m4.modules()):
    for name, sub in list(mod.named_children()):
        if isinstance(sub, LoRALinear):
            with torch.no_grad():
                setattr(mod, name, Linear4(sub.base.weight + (sub.B @ sub.A) * sub.scale))
use["합친 뒤 다시 4비트"] = evaluate(m4); del m4
for k, v in use.items():
    print(f"{k}:", brief(v))
del model

## 11.5 학습 한 걸음의 메모리

In [ ]:
def footprint(kind):
    """학습 한 걸음(배치 8)에서 잡힌 텐서를 종류별로 센다. 역전파를 위해 남겨 두는 텐서는 saved_tensors_hooks로 센다."""
    if kind == "전체 미세 조정":
        model = load(); params = list(model.parameters())
    else:
        model = lora_model(ATTN, 8, quant=kind.startswith("QLoRA"))
        params = [p for p in model.parameters() if p.requires_grad]
    Linear4.recompute = kind == "QLoRA(다시 되돌림)"
    model.train()
    opt = torch.optim.AdamW(params, lr=1e-5, weight_decay=0.0)
    held = {t.untyped_storage().data_ptr(): t.untyped_storage().nbytes() for t in list(model.parameters()) + list(model.buffers())}
    rows = [example(t, y) for t, y in TRAIN[:8]]
    L = max(len(r[0]) for r in rows)
    ids = torch.tensor([r[0] + [tok.pad_token_id] * (L - len(r[0])) for r in rows])
    lab = torch.tensor([r[1] + [-100] * (L - len(r[1])) for r in rows])
    mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rows])
    saved = {}
    def pack(t):                                                 # 가중치가 아닌 텐서만 센다(같은 저장소는 한 번)
        s = t.untyped_storage()
        if s.data_ptr() not in held:
            saved[s.data_ptr()] = s.nbytes()
        return t
    lap = time.perf_counter()
    for step in range(2):
        with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
        if step == 0:
            kept = sum(saved.values())
        opt.zero_grad(); loss.backward(); opt.step()
    GB = 1e9
    out = {"가중치": sum(held.values()) / GB, "기울기": sum(p.grad.nbytes for p in params) / GB,
           "AdamW 상태": sum(v.nbytes for st in opt.state.values() for v in st.values() if torch.is_tensor(v) and v.dim() > 0) / GB,
           "역전파용 텐서": kept / GB, "걸음 초": (time.perf_counter() - lap) / 2, "배치 토큰": int(mask.sum()), "배치 길이": L}
    Linear4.recompute = False
    del model, opt, params
    return out

mem = {}
for kind in ["전체 미세 조정", "LoRA", "QLoRA", "QLoRA(다시 되돌림)"]:
    mem[kind] = footprint(kind)
    m = mem[kind]
    print(f"{kind}: " + ", ".join(f"{k} {m[k]:.3f}GB" for k in ["가중치", "기울기", "AdamW 상태", "역전파용 텐서"])
          + f", 합계 {sum(m[k] for k in ['가중치', '기울기', 'AdamW 상태', '역전파용 텐서']):.2f}GB, 한 걸음 {m['걸음 초']:.1f}초")
print("배치:", mem["LoRA"]["배치 길이"], "칸 ×8, 실제 토큰", mem["LoRA"]["배치 토큰"])
grads = []                                                       # 두 방식의 기울기가 같은지 확인한다
for flag in [False, True]:
    Linear4.recompute = flag
    m = lora_model(ATTN, 8, quant=True); m.train()
    m(input_ids=x, labels=x).loss.backward()
    grads.append(torch.cat([p.grad.flatten() for p in m.parameters() if p.requires_grad]))
    del m
Linear4.recompute = False
print("두 방식의 LoRA 기울기 차이(최대):", (grads[0] - grads[1]).abs().max().item(), "기울기 크기(최대):", round(grads[0].abs().max().item(), 4))

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
names = list(runs)
last = [runs[k][-1] for k in names]
fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.4), dpi=260, gridspec_kw={"height_ratios": [1.45, 1]})
y = np.arange(len(names))[::-1] * 1.1
for off, key, color, label in [(0.3, "acc", BLUE, "영화평 정확도"), (0, "qa", GREEN, "상식 정답"), (-0.3, "other_pn", ORANGE, "다른 지시에 긍정·부정")]:
    axes[0].barh(y + off, [e[key] * 100 for e in last], height=0.28, color=color, label=label)
    for yy, e in zip(y, last):
        axes[0].text(e[key] * 100 + 1, yy + off, f"{e[key] * 100:.1f}", va="center", fontsize=8)
axes[0].set_yticks(y, names); axes[0].set_xlim(0, 100); axes[0].set_xlabel("%", fontsize=8)
axes[0].set_title("① 3에폭 뒤의 시험", loc="left", fontsize=8.6, color="#374151")
axes[0].legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.4, -0.13), ncol=3, handlelength=1.2, columnspacing=1.0)
kinds = list(mem)
xx = np.arange(len(kinds)); bottom = np.zeros(len(kinds))
for key, color in [("가중치", GRAY), ("기울기", RED), ("AdamW 상태", ORANGE), ("역전파용 텐서", PURPLE)]:
    v = np.array([mem[k][key] for k in kinds])
    axes[1].bar(xx, v, 0.55, bottom=bottom, color=color, label=key); bottom += v
for i, t in enumerate(bottom):
    axes[1].text(i, t + 0.2, f"{t:.2f}", ha="center", fontsize=8)
axes[1].set_xticks(xx, ["전체 미세 조정", "LoRA", "QLoRA", "QLoRA\n(다시 되돌림)"]); axes[1].set_ylabel("GB", fontsize=8)
axes[1].set_ylim(0, 14)
axes[1].set_title("② 학습 한 걸음의 메모리(배치 8)", loc="left", fontsize=8.6, color="#374151")
axes[1].legend(frameon=False, fontsize=8, loc="upper right", handlelength=1.2)
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_11_lora.png"); plt.close(fig)